# Dinâmica 05 - Otimização Evolutiva de Agentes Autônomos (AG Cars)
**Unidade Curricular:** Inteligência Artificial  
**Câmpus:** Chapecó  
**Docente:** Dra. Lara Popov Zambiasi Bazzi Oberderfer  

---
## 🎯 Objetivo
Utilizar **Algoritmos Genéticos** para treinar a rede neural de uma população de carrinhos autônomos. O objetivo é que os agentes aprendam a navegar em uma pista reta, utilizando sensores de proximidade para evitar colisões com as paredes laterais. Observaremos a evolução visual do comportamento da "burrice" inicial para a pilotagem precisa.

In [13]:
# = ================================================================
# 1. PREPARAÇÃO DO AMBIENTE E BIBLIOTECAS GRÁFICAS
# =================================================================
# Instalação de dependências para renderizar vídeo no Colab
!apt-get install -y xvfb python-opengl ffmpeg > /dev/null 2>&1
!pip install pyvirtualdisplay > /dev/null 2>&1

import numpy as np
import random
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML
from pyvirtualdisplay import Display

# Inicializa o monitor virtual para capturar os gráficos
d = Display(visible=0, size=(1024, 768))
d.start()

print("Ambiente gráfico configurado. Pronto para a evolução visual.")

Ambiente gráfico configurado. Pronto para a evolução visual.


### 🧠 Sprint 1: Definição da Física e do "Cérebro"
O carrinho possui 3 sensores de proximidade (Esquerda, Frente, Direita) que medem a distância até a parede. O "cérebro" é uma rede neural simples que recebe esses 3 sinais e decide o ângulo de esterçamento da direção. O Algoritmo Genético otimizará os **pesos** dessa rede neural.

In [14]:
# = ================================================================
# 2. FÍSICA DO CARRINHO, PISTA E REDE NEURAL (CÉREBRO)
# =================================================================

# Configurações da Pista
LARGURA_PISTA = 100
COMPRIMENTO_PISTA = 1000
PAREDE_ESQ = 0
PAREDE_DIR = LARGURA_PISTA

class Carro:
    def __init__(self, weights):
        self.x = LARGURA_PISTA / 2  # Posição lateral inicial (centro)
        self.y = 0                 # Posição longitudinal inicial
        self.angulo = 90           # Apontando para frente (90 graus)
        self.velocidade = 5        # Velocidade constante
        self.alive = True
        self.fitness = 0
        self.pesos = weights       # Os pesos da rede neural (o cromossomo)

        # Histórico de trajetória para o desenho
        self.trajetoria_x = [self.x]
        self.trajetoria_y = [self.y]

    def reset(self):
        self.__init__(self.pesos)

    def ler_sensores(self):
        """Simula 3 sensores de distância (Esq, Frente, Dir)"""
        dist_esq = self.x - PAREDE_ESQ
        dist_dir = PAREDE_DIR - self.x
        dist_frente = COMPRIMENTO_PISTA - self.y # Pista reta, frente é simples

        # Normalização simples (0 a 1) baseada na largura da pista
        return np.array([dist_esq/LARGURA_PISTA, dist_frente/COMPRIMENTO_PISTA, dist_dir/LARGURA_PISTA])

    def cérebro_rede_neural(self, sensores):
        """Rede Neural Simples (MLP): Entrada(3) -> Saída(1)"""
        # Entrada: [Esq, Frente, Dir]
        # Pesos: [w_esq, w_frente, w_dir, bias] (4 pesos no total)

        # Ativação Linear Simples (Soma Ponderada)
        soma = np.dot(sensores, self.pesos[:3]) + self.pesos[3]

        # Saída: Tensão de esterçamento (Mapeada para -30 a +30 graus)
        # Usamos Tanh para limitar a saída entre -1 e 1
        esterçamento = np.tanh(soma) * 30
        return esterçamento

    def update(self):
        """Atualiza a física do carrinho baseada na decisão do cérebro"""
        if not self.alive: return

        sensores = self.ler_sensores()
        esterçamento = self.cérebro_rede_neural(sensores)

        # Atualiza o ângulo do carro
        self.angulo += esterçamento

        # Limita o ângulo para não dar ré (entre 45 e 135 graus)
        self.angulo = np.clip(self.angulo, 45, 135)

        # Move o carro (trigonometria básica)
        rad = np.radians(self.angulo)
        self.x += self.velocidade * np.cos(rad)
        self.y += self.velocidade * np.sin(rad)

        # Registra a trajetória
        self.trajetoria_x.append(self.x)
        self.trajetoria_y.append(self.y)

        # Verificação de Colisão (Fitness cai para 0 se bater)
        if self.x <= PAREDE_ESQ or self.x >= PAREDE_DIR or self.y >= COMPRIMENTO_PISTA:
            self.alive = False

    def calcular_fitness(self):
        """A aptidão é baseada na distância percorrida na pista"""
        # Se bateu muito cedo, ganha pouca aptidão
        self.fitness = self.y
        if not self.alive and self.y < COMPRIMENTO_PISTA * 0.9:
            self.fitness *= 0.5 # Penalidade por colisão precoce
        return self.fitness

print("Física dos carrinhos e Cérebro Neural definidos.")

Física dos carrinhos e Cérebro Neural definidos.


### 🧬 Sprint 2: Ciclo Genético
Nesta etapa, rodamos o algoritmo genético por algumas gerações. A "evolução" dos pesos da rede neural fará com que os carrinhos passem de movimentos aleatórios para uma pilotagem centralizada na pista.

In [15]:
# = ================================================================
# 3. ALGORITMO GENÉTICO E EVOLUÇÃO DO CÉREBRO
# =================================================================

# Hiperparâmetros do AG
TOTAL_CARRINHOS = 50
GERAÇÕES = 30
TAXA_MUTACAO = 0.1 # 10% de chance de alterar um peso
QTD_PESOS = 4      # [w_esq, w_frente, w_dir, bias]

# Inicialização da População Aleatória (Pesos entre -1 e 1)
populacao_pesos = [np.random.uniform(-1, 1, QTD_PESOS) for _ in range(TOTAL_CARRINHOS)]

history_fitness = []

print(f"Iniciando evolução de {TOTAL_CARRINHOS} cérebros por {GERAÇÕES} gerações...")

for g in range(1, GERAÇÕES + 1):
    # Cria os carrinhos com os cérebros da geração atual
    carrinhos = [Carro(pesos) for pesos in populacao_pesos]

    # --- SIMULAÇÃO DA GERAÇÃO (Rodar até todos baterem ou acabar a pista) ---
    for _ in range(200): # Limite de passos na pista reta
        vivos = 0
        for carro in carrinhos:
            if carro.alive:
                carro.update()
                vivos += 1
        if vivos == 0: break # Todos bateram, fim da geração

    # --- AVALIAÇÃO E SELEÇÃO ---
    aptidões = [carro.calcular_fitness() for carro in carrinhos]
    history_fitness.append(np.max(aptidões))

    # Elitismo: Encontrar o melhor cérebro para sobreviver
    idx_melhor = np.argmax(aptidões)
    melhor_cérebro = populacao_pesos[idx_melhor]

    # --- REPRODUÇÃO (Criar nova geração) ---
    nova_populacao_pesos = [melhor_cérebro] # O melhor sobrevive sem alteração

    while len(nova_populacao_pesos) < TOTAL_CARRINHOS:
        # Seleção dos Pais (Torneio Simples)
        idx_p1 = random.randint(0, TOTAL_CARRINHOS - 1)
        idx_p2 = random.randint(0, TOTAL_CARRINHOS - 1)
        pai1 = populacao_pesos[idx_p1]
        pai2 = populacao_pesos[idx_p2]

        # Crossover Uniforme (Mistura os pesos dos pais)
        filho = [random.choice([pai1[i], pai2[i]]) for i in range(QTD_PESOS)]
        nova_populacao_pesos.append(np.array(filho))

    # --- MUTAÇÃO ---
    for i in range(1, TOTAL_CARRINHOS): # Pula o elite
        if random.uniform(0, 1) < TAXA_MUTACAO:
            # Altera aleatoriamente um dos 4 pesos
            gene_idx = random.randint(0, QTD_PESOS - 1)
            populacao_pesos[i][gene_idx] += np.random.normal(0, 0.2) # Adiciona ruído gaussiano

    if g % 10 == 0:
        print(f"Geração {g}/{GERAÇÕES} | Melhor Distância: {np.max(aptidões):.1f}m")

print("\nEvolução concluída!")
print(f"Pesos do Cérebro Campeão: {melhor_cérebro}")

Iniciando evolução de 50 cérebros por 30 gerações...
Geração 10/30 | Melhor Distância: 993.0m
Geração 20/30 | Melhor Distância: 987.5m
Geração 30/30 | Melhor Distância: 988.5m

Evolução concluída!
Pesos do Cérebro Campeão: [ 0.35878608 -0.24097613 -0.99342119  0.63478267]


### 🎬 Sprint 3: Demonstração Visual (A Evolução em Vídeo)
Vamos gerar uma animação que mostra a trajetória de todos os carrinhos na Geração 1 (burros) e compará-la com a Geração Final (treinados). Isso valida visualmente o poder da otimização evolutiva.

In [16]:
# = ================================================================
# 4. GERAÇÃO DA ANIMAÇÃO VISUAL E GRÁFICO DE FITNESS (CARRINHOS EVOLUINDO)
# =================================================================

# --- SIMULAÇÃO FINAL ---
# Rodar uma simulação final com os cérebros campeões para desenhar a trajetória
carrinhos_finais = [Carro(pesos) for pesos in populacao_pesos]
passos_finais = 200
history_final_vivos = []

for p in range(passos_finais):
    vivos = 0
    for carro in carrinhos_finais:
        if carro.alive:
            carro.update()
            vivos += 1
    history_final_vivos.append(vivos) # Quantos carros estão vivos em cada passo

# --- CONFIGURAÇÃO DA FIGURA DUPLA (PISTA + FITNESS) ---
# Criamos uma figura com 2 subplots (1 linha, 2 colunas)
fig, (ax_pista, ax_fitness) = plt.subplots(1, 2, figsize=(14, 10), gridspec_kw={'width_ratios': [1, 1.5]})

# 1. Configuração do Subplot da Pista (Esquerda)
ax_pista.set_xlim(PAREDE_ESQ - 10, PAREDE_DIR + 10)
ax_pista.set_ylim(-10, COMPRIMENTO_PISTA + 10)
ax_pista.axvline(x=PAREDE_ESQ, color='black', linewidth=3, label='Parede Esq')
ax_pista.axvline(x=PAREDE_DIR, color='black', linewidth=3, label='Parede Dir')
ax_pista.set_title("Visualização Final: População Treinada")
ax_pista.set_xlabel("Posição Lateral (X)")
ax_pista.set_ylabel("Distância Percorrida (Y)")
ax_pista.legend(loc='lower center')
# Linhas de trajetória para cada carrinho na pista
linhas_pista = [ax_pista.plot([], [], alpha=0.3)[0] for _ in range(TOTAL_CARRINHOS)]

# 2. Configuração do Subplot de Fitness (Direita)
ax_fitness.set_title('Evolução do Treinamento: Q-Learning')
ax_fitness.set_xlabel('Gerações')
ax_fitness.set_ylabel('Melhor Distância (Aptidão Máxima)')
ax_fitness.grid(True, linestyle='--')
ax_fitness.set_xlim(0, GERAÇÕES)
ax_fitness.set_ylim(0, COMPRIMENTO_PISTA * 1.1)
# Linha do gráfico de aptidão
linha_fitness, = ax_fitness.plot([], [], color='red', label='Melhor Distância (Aptidão Máxima)')
ax_fitness.legend()

# Texto dinâmico para mostrar as informações
txt_info = ax_pista.text(10, 50, '', color='black', fontsize=12)

# --- FUNÇÕES DE ANIMAÇÃO ---
def init():
    """Inicializa as linhas e textos para o primeiro frame"""
    # Inicializa as trajetórias na pista
    for linha in linhas_pista:
        linha.set_data([], [])

    # Inicializa o gráfico de fitness (estático durante a animação dos carros)
    linha_fitness.set_data(range(len(history_fitness)), history_fitness)

    # Inicializa o texto
    txt_info.set_text('')

    return linhas_pista + [linha_fitness, txt_info]

def animate(i):
    """Função de animação frame a frame"""

    # 1. Atualiza as trajetórias na pista até o passo 'i'
    for idx, carro in enumerate(carrinhos_finais):
        # Mapeia os índices de trajetória até o frame 'i'
        x = carro.trajetoria_x[:i]
        y = carro.trajetoria_y[:i]
        linhas_pista[idx].set_data(x, y)

        # Destaca o melhor carrinho (elite) em vermelho
        if idx == 0:
            linhas_pista[idx].set_alpha(1.0)
            linhas_pista[idx].set_color('red')
            linhas_pista[idx].set_linewidth(3)
        else:
             # Deixa os outros mais transparentes se já tiverem batido
            if not carro.alive and len(y) < i:
                linhas_pista[idx].set_alpha(0.1)

    # 2. Atualiza o texto dinâmico (Passo e quantos carros ainda estão vivos)
    txt_info.set_text(f"Passo: {i+1}\nCarros Vivos: {history_final_vivos[min(i, passos_finais-1)]}")

    # 3. O gráfico de fitness (direita) não muda frame a frame nesta animação
    # Ele mostra o resultado de todas as gerações anteriores de uma vez.

    # Retorna todos os artistas que foram modificados
    return linhas_pista + [linha_fitness, txt_info]

# --- CRIAÇÃO E EXIBIÇÃO DA ANIMAÇÃO ---
# Cria a animação (mostrando os passos da simulação final)
ani = animation.FuncAnimation(fig, animate, init_func=init, frames=passos_finais, interval=50, blit=True)

# Converte a animação Matplotlib para HTML5 Vídeo para exibir no Colab
print("Gerando vídeo da simulação final e gráfico... (Aproximadamente 30 segundos)")
plt.close() # Impede que o gráfico estático apareça
HTML(ani.to_html5_video())

Gerando vídeo da simulação final e gráfico... (Aproximadamente 30 segundos)


# 📝 Ficha de Análise: Dinâmica 05 (AG Cars)

### Reflexão Crítica e Visual:

1. **Análise Visual:** Assista ao vídeo gerado. Como você descreveria a trajetória da maioria dos carrinhos da geração final? Eles andam em linha reta ou oscilam?
2. **Fitness:** No nosso código, a função fitness é baseada apenas na distância percorrida (`self.y`). Se mudássemos para `self.y / (passos_dados)`, o algoritmo priorizaria carrinhos mais rápidos ou carrinhos que sobrevivem mais tempo?
3. **Cérebro Neural:** Os "pesos" que o AG otimizou são, na prática, os ganhos de um controlador. O peso `w_esq` (ligado ao sensor esquerdo) deve ser positivo ou negativo para que o carro vire para a **direita** ao se aproximar da parede esquerda?
4. **Mutação:** O que aconteceria visualmente se definíssemos a `TAXA_MUTACAO = 0.9` (90%)? O comportamento da população ficaria mais estável ou mais caótico a cada geração?
5. **Aplicação na Engenharia:** Como Engenheiro(a) de Automação, como você utilizaria essa técnica para otimizar a trajetória de um braço robótico que precisa desviar de obstáculos fixos?